# SAND: every solve folded into the optimiser

SAND (simultaneous analysis and design) hands everything to the optimiser. Its unknowns
are the iteration variables plus every quantity that a solve inside the models would
otherwise determine: the copies that open the feedback loops, the coil width from its
root find, the ion temperature, and so on. Each of those solves becomes an equality
constraint next to the file's own. There is no inner loop. One SQP runs over one block,
each evaluation is one pass of the models, and the models agree with each other only at
the optimum.

SAND takes four operations on the graph. Cut the feedback loops. Insert the
optimisation problem. Turn every fixed-point requirement into a residual
(`Residualise`: `x = g(x)` becomes `g(x) - x = 0`). Merge every problem in the loop
into one (`Combine`). The result is what `sand.assemble` builds and what `session`'s SAND
arm solves. This notebook spells the steps out, shows the run
order, and runs it.

In [1]:
import os, sys, time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
# The editable cottax checkout beside this repo (`../jaxgraph/src`), ahead of any
# installed copy.
_jaxgraph = REPO.parent.parent / "jaxgraph" / "src"
for entry in (str(REPO), str(_jaxgraph)):
    if Path(entry).is_dir() and entry not in sys.path:
        sys.path.insert(0, entry)

import jax
jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
import cottax
import numpy as np


print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). One disconnected node is left out. The file also poses the
problem: iteration variables `ixc`, constraints `icc`, figure of merit.
`configurations/stellarator_helias.py` states all of that as a tree -- the input
file converted once -- and `native.reference_of` reads it without running PROCESS.

In [2]:
CONFIGURATION = "stellarator_helias"                   # configurations/stellarator_helias.py: the machine, its values, its problem

import re

from cottax.blocking import Blocking
from cottax.plan import Plan
from cottax.problem import is_fixed_point

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.cottax.input import native
from functional_process.configurations import load
from functional_process.cottax.input.indat import graph_for
from functional_process.cottax.architectures.evaluate import without_excluded
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

configuration = load(CONFIGURATION)
ref = native.reference_of(configuration)      # ixc, icc, bounds, cold values -- PROCESS-free
sw = configuration.problem.switches           # the static switch values the condition nodes are bound with
machine_graph = graph_for(configuration.machine)
raw = without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")
print(f"\nixc = {ref.ixc}")
print(f"icc = {ref.icc}  ({ref.n_equality} equalities), objective: figure of merit {ref.i_figure_merit}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point

ixc = [2, 3, 4, 6, 10, 56, 59, 109]
icc = [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (2 equalities), objective: figure of merit 6


## The recipe

### Step 1: cut the feedback loops

Each group of models that feed back into each other is opened with copies. `mda.CUTS`
names nine variables, picked by hand so that one iteration of the copies is one PROCESS
pass. `mda.cut_ops` turns the ones present in this graph into one operation per loop.
Each `FixedPointCut` gives the readers of a variable a copy, `^hat.x`, and adds the
requirement that the copy equals the computed value. That requirement is a fixed-point
problem. The `mda_gauss_seidel` notebook derives such cuts mechanically, and a recipe
from there can be used here instead.

In [3]:
from functional_process.cottax.architectures.mda import cut_graph, cut_ops

plan = Plan(raw)
for op in cut_ops(raw):
    plan = plan + op
print_recipe(plan)
print("\nsame as mda.cut_graph(raw):", plan.graph == cut_graph(raw))
minted = [p for p in declared(plan.graph) if p not in set(declared(raw))]
print("problems the cut minted:", [p.spelling for p in minted])

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)

same as mda.cut_graph(raw): True
problems the cut minted: ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']


### Step 2: state the file's problem

One node per active constraint and one for the figure of merit. They compute the same
values `constraints.py` and `objectives.py` do. Then the optimisation problem itself,
an `Optimise` node: minimise the objective over the iteration variables, subject to the
constraints. All are inserted into the graph like any other node. `sand.optimise_graph`
does the same; it is spelled out here so the step is visible.

In [4]:
from cottax.names import PathMap
from cottax.plan import Insert
from cottax.problem import Optimise
from cottax.spec import NodePath
from jax.tree_util import GetAttrKey

from functional_process.cottax.input.indat import objective_selection
from functional_process.cottax.architectures.sand import constraint_nodes, iteration_variable_path, objective_nodes, optimise_graph

nodes, equalities, inequalities, omitted = constraint_nodes(plan.graph, ref.icc, ref.n_equality, sw)
objective_built, objective = objective_nodes(plan.graph, objective_selection(ref.i_figure_merit), sw)
nodes.update(objective_built)
OPT = NodePath((GetAttrKey("Opt"),))
nodes[OPT] = Optimise(
    objective=objective,
    unknowns=tuple(iteration_variable_path(i) for i in ref.ixc),
    equalities=tuple(equalities),
    inequalities=tuple(inequalities),
)
plan = plan + Insert(PathMap(nodes.items()))
print_recipe(plan)
if omitted:
    print("constraints this port cannot state yet:", omitted)
reference_graph, _, _ = optimise_graph(cut_graph(raw), ref.ixc, ref.icc, ref.n_equality, ref.i_figure_merit, switch_values=sw)
print("same as sand.optimise_graph(...):", plan.graph == reference_graph)

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
same as sand.optimise_graph(...): True


The optimisation problem reads the objective and the constraints. It also owns the
iteration variables, which almost everything reads. So it closes one big loop over most
of the machine. That loop now contains several problems. The blocking refuses it until
it is told how they relate, and the refusal names the two options: `Combine` and
`NestInside`. Choosing between them is choosing the architecture.

In [5]:
from cottax.partition import OrderedPartition

partition = OrderedPartition.scc(plan.graph)          # the components, with no claim that they can be run
big = max(partition.blocks, key=len)
print(f"largest block: {len(big)} of {len(plan.graph.nodes)} nodes, holding",
      [p.spelling for p in declared(plan.graph) if p in set(big)], "\n")
try:
    Blocking.scc(plan.graph)                          # a blocking is a partition a schedule could be built on
except ValueError as refusal:                 # the node list dropped from the message
    print(re.sub(r"block \(.*?\) declares", "the block declares", str(refusal), flags=re.S))

largest block: 123 of 170 nodes, holding ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single', '.Opt'] 



the block declares several problems ((NodePath(^problem.stellarator.coils.intersect), NodePath(^problem.physics.profiles.ion_vol_avg_temperature), NodePath(^problem.physics.proton_rate_density.cycle), NodePath(^problem.fwbs.f_ster_div_single), NodePath(.Opt))) with nothing saying which is outer -- one driver answers one problem, so `Combine` them into a single problem over every unknown, or `NestInside` one of them. Which is a modelling decision, not something the blocking can read off the graph


### Step 3: residualise, then combine

Before merging, `sand.assemble` runs the analysis once (`mda_env`). It then
drops any fixed-point requirement that is degenerate there (identically zero) or
array-valued, because neither can be an SQP unknown. This file has none, so the check is
shown and no `Delete` is needed. The analysis result is kept: the solve starts from it.

In [6]:
from cottax.plan import Delete

from functional_process.cottax.architectures.sand import array_valued_problems, degenerate_fixed_points
from functional_process.cottax.architectures.evaluate import mda_env

driven, env = mda_env(ref, graph=machine_graph)          # one converged MDA on the cut graph
drop = tuple(degenerate_fixed_points(driven, env)) + tuple(array_valued_problems(driven, env))
print("fixed points to drop before folding:", [p.spelling for p in drop] or "none")
if drop:
    plan = plan + Delete(drop)

fixed points to drop before folding: none


In [7]:
from cottax.rewrites import Combine, Residualise

from functional_process.cottax.architectures.sand import assemble

for p in declared(plan.graph):
    if is_fixed_point(plan.graph[p]):
        plan = plan + Residualise(p)                     # `x = g(^hat.x)`  ->  `g(^hat.x) - x = 0`

SAND = NodePath((GetAttrKey("sand"),))
folding = (OPT, *[p for p in declared(plan.graph) if p != OPT])   # the optimiser first: its unknowns lead
plan = plan + Combine(SAND, folding)

print("the whole recipe:")
print_recipe(plan)

reference, report = assemble(ref, driven, env, switch_values=sw)
print("\nsame as sand.assemble(...):", plan.graph == reference)
blocking = Blocking.scc(plan.graph)
print("problems, in run order:", [p.spelling for p in blocking.problems if p is not None])

the whole recipe:
   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
   residualise(^problem.physics.profiles.ion_vol_avg_temperature)
   residualise(^problem.power.delta_eta_step)
   residualise(^problem.physics.proton_rate_density.cycle)
   residualise(^problem.fwbs.f_ster_div_single)
   combine(^problem.sand <- .Opt, ^problem.stellarator.coils.intersect, ^problem.physics.profiles.ion_vol_avg_tem ...



same as sand.assemble(...): True
problems, in run order: ['^problem.sand']


One problem where there were five. Its unknowns are the eight iteration variables plus
six lifted from the merged solves. Its conditions are the objective, the fourteen
constraints and one residual per lifted unknown.

### Assign the driver

VMCON on the merged problem, attached in the graph by `sand.sand_schedule`. The one root
find left outside the loop keeps its Newton.

In [8]:
from functional_process.cottax.architectures.sand import sand_schedule, sand_shape

schedule = sand_schedule(plan.graph, None, bounds=ref.bounds)   # `Assign` VMCON on `^problem.sand`, Newton on the rest
shape = sand_shape(schedule)
drive = shape["drive"]
print(f"the SAND block: {shape['drive_nodes']} nodes, {shape['unknowns']} unknowns "
      f"({shape['design']} design), {shape['conditions']} conditions "
      f"({shape['equalities']} equalities, {shape['inequalities']} inequalities); "
      f"{shape['schedule_steps']} schedule steps in all")
print("unknowns:", [u.spelling for u in drive.unknowns])

the SAND block: 124 nodes, 14 unknowns (14 design), 21 conditions (8 equalities, 12 inequalities); 46 schedule steps in all
unknowns: ['.physics.b_plasma_toroidal_on_axis', '.physics.rmajor', '.physics.temp_plasma_electron_vol_avg_kev', '.physics.nd_plasma_electrons_vol_avg', '.physics.hfact', '.tfcoil.t_tf_superconductor_quench', '.tfcoil.f_a_tf_turn_cable_copper', '.physics.f_nd_alpha_thermal_electron', '.stellarator.wp_width_r_min', '.physics.temp_plasma_ion_vol_avg_kev', '.power.delta_eta', '^hat.physics.proton_rate_density', '^hat.physics.fusden_alpha_total', '^hat.fwbs.f_ster_div_single']


## The process

The DSM in run order. One box surrounds the whole coupled block, VMCON on it, and there
is no box inside it. Every former inner solve is now a row of residual conditions the
optimiser answers. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [9]:
from functional_process.cottax.visualization.render_xdsm import SPELLING
from functional_process.cottax.visualization.grouping import render_grouped_dsm_html, structure_order

blocking = schedule.blocking
dsm = render_grouped_dsm_html(
    blocking, order=structure_order(blocking),
    title="stellarator_helias -- SAND: every solve combined into one VMCON block",
    file_name="dsm_sand", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch


written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/sand/dsm_sand.html


## Run it

As `session.solve_block` does it. The driver records each iterate and scales each
residual by the size of its quantity, so a residual on a value of order `1e17` is
brought to order one. The block starts with the iteration variables at the input file's
values and every lifted unknown at the converged analysis. The schedule then runs step
by step, because VMCON runs outside the compiled program.

In [10]:
from functional_process.cottax.architectures import mdf
from functional_process.cottax.architectures.drivers import Status
from functional_process.cottax.architectures.session import recorder, trace_tail
from functional_process.cottax.architectures.evaluate import inputs_only, seed_block
from functional_process.cottax.architectures.session import SAND_MAX_ITER
from functional_process.cottax.architectures.sand import residual_condition_scales
from functional_process.cottax.architectures.evaluate import run_schedule

trace = []
solve = sand_schedule(plan.graph, None, bounds=ref.bounds,
                      condition_scale=residual_condition_scales(drive, env),
                      callback=recorder(trace), max_iter=SAND_MAX_ITER)
solve_drive = sand_shape(solve)["drive"]
design_vars = {iteration_variable_path(i) for i in ref.ixc}
seeded, borrowed = seed_block(solve, solve_drive, ref.cold, env, design=design_vars)

began = time.perf_counter()
out = run_schedule(solve, inputs_only(solve, seeded), whole=False)
print(f"solved in {time.perf_counter() - began:.1f} s (first solve: includes compilation)")

status = int(np.asarray(mdf.verdict(out, Status, solve_drive.problem)))
iterations, objf, max_eq, min_ie = trace_tail(trace)
print(f"VMCON: status {status}, {iterations} iterations")
print(f"objective {objf:.8f}   max|eq| {max_eq:.1e}   min ineq {min_ie:+.1e}")
print("design:", {i: round(float(np.asarray(out[iteration_variable_path(i)])), 4) for i in ref.ixc})

trace.clear()
began = time.perf_counter()
out = run_schedule(solve, inputs_only(solve, seeded), whole=False)
print(f"\nwarm solve: {time.perf_counter() - began:.2f} s, {len(trace)} iterations")

solved in 8.5 s (first solve: includes compilation)
VMCON: status 0, 43 iterations
objective 1.21844143   max|eq| 1.2e-06   min ineq -4.3e-10
design: {2: 4.7164, 3: 26.6445, 4: 5.7027, 6: 1.7391774970059938e+20, 10: 1.048, 56: 31.8104, 59: 0.7177, 109: 0.0299}



warm solve: 0.92 s, 43 iterations


The same solve is one line: `session.open_session("stellarator_helias").sand()`. `tests/test_architectures.py`
pins it beside the other arms on every regression input file.

In [11]:
RESULT = {"status": status, "iterations": iterations, "objf": objf, "max_eq": max_eq, "min_ie": min_ie,
          "design": {i: float(np.asarray(out[iteration_variable_path(i)])) for i in ref.ixc},
          "unknowns": len(solve_drive.unknowns), "conditions": len(solve_drive.conditions)}
RESULT

{'status': 0,
 'iterations': 43,
 'objf': 1.218441432897411,
 'max_eq': 1.2305890117370283e-06,
 'min_ie': -4.3244252623253487e-10,
 'design': {2: 4.716449671977104,
  3: 26.64445535650798,
  4: 5.702734340773733,
  6: 1.7391774970059938e+20,
  10: 1.047952379234968,
  56: 31.810415391262385,
  59: 0.7176935320205386,
  109: 0.029925036975006085},
 'unknowns': 14,
 'conditions': 21}